# Bước 4 — Lưu vector vào FAISS
Notebook tạo vector index thật bằng FAISS CPU. FAISS giữ vector; `chunks.json` giữ nội dung và metadata ở cùng thứ tự.

In [ ]:
from pathlib import Path
import json
import faiss
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
STORAGE_DIR = PROJECT_DIR / 'storage'
chunks = json.loads((STORAGE_DIR / 'chunks.json').read_text(encoding='utf-8'))
embeddings = np.load(STORAGE_DIR / 'embeddings.npy').astype('float32')
print('Số chunk:', len(chunks))
print('Shape embeddings:', embeddings.shape)

## Tạo index
Vector đã được chuẩn hóa ở bước 3. Vì vậy `IndexFlatIP` cho điểm tương đương cosine similarity. Flat index tìm chính xác và phù hợp dataset nhỏ.

In [ ]:
if len(chunks) != len(embeddings):
    raise ValueError('Số chunk và vector không khớp')

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print('Số chiều:', index.d)
print('Số vector:', index.ntotal)

## Thử truy vấn trực tiếp
FAISS trả về hai mảng: điểm tương đồng và vị trí vector. Vị trí đó dùng để lấy metadata trong `chunks`.

In [ ]:
scores, indices = index.search(embeddings[0:1], 3)
print('Điểm:', scores[0])
print('Vị trí:', indices[0])
print('Nguồn gần nhất:', chunks[int(indices[0][0])]['source'])

In [ ]:
output = STORAGE_DIR / 'faiss.index'
faiss.write_index(index, str(output))
print('Đã lưu:', output)